In [1]:
# Cell 1 - Install everything we need
!pip install mesa rdflib streamlit pandas

   ---------------------------------------- 0.0/615.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/615.4 kB ? eta -:--:--
   ---------------------------------- ----- 524.3/615.4 kB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 615.4/615.4 kB 2.0 MB/s  0:00:00


In [2]:
# Cell 2 - Quick check that the ontology file is readable
from rdflib import Graph

g = Graph()
g.parse("lab_ontology.ttl", format="turtle")
print("Ontology loaded! Total facts:", len(g))

Ontology loaded! Total facts: 66


In [4]:
# Cell 3 (fixed) - Imports for Mesa 3.x
from mesa import Agent, Model
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, OWL, XSD
from datetime import datetime

NS = Namespace("http://example.org/lab.owl#")

In [9]:
# Cell 4 (fixed for Mesa 3) - StudentAgent
class StudentAgent(Agent):
    def __init__(self, model, unique_id, student_uri, equipment_name, time_slot, duration_hours):
        super().__init__(model)                 # Mesa 3: only pass model
        self.unique_id = unique_id              # set id manually
        self.student_uri = student_uri          # e.g. NS["Student1"]
        self.equipment_name = equipment_name    # e.g. "LaserCutter1"
        self.time_slot = time_slot              # e.g. "Mon10"
        self.duration_hours = duration_hours
        self.result = None                      # will store the final answer

    def step(self):
        # Ask the lab manager to process our request
        self.result = self.model.lab_manager.process_request(self)

In [13]:
# Cell 5 (fixed) - LabManagerAgent
class LabManagerAgent(Agent):
    def __init__(self, model, unique_id, graph):
        super().__init__(model)
        self.unique_id = unique_id
        self.graph = graph
        self.reservation_counter = 0
        self.log = []

    def step(self):
        # The manager does not act on its own.
        # It only responds when a student calls process_request().
        pass

    def process_request(self, student):
        eq_uri = NS[student.equipment_name]
        stu_uri = student.student_uri

        # ---- CHECK 1: Equipment exists ----
        if (eq_uri, RDF.type, NS.Equipment) not in self.graph:
            return self._deny(student, "Equipment not found in ontology")

        # ---- CHECK 2: Under maintenance ----
        for m in self.graph.objects(eq_uri, NS.underMaintenance):
            if m == Literal(True, datatype=XSD.boolean):
                return self._deny(student, "Equipment is under maintenance")

        # ---- CHECK 3: Available ----
        for a in self.graph.objects(eq_uri, NS.isAvailable):
            if a != Literal(True, datatype=XSD.boolean):
                return self._deny(student, "Equipment is not available")

        # ---- CHECK 4: Certification ----
        required_certs = list(self.graph.objects(eq_uri, NS.requiresCertification))
        if required_certs:
            student_certs = list(self.graph.objects(stu_uri, NS.hasCertification))
            if not any(cert in student_certs for cert in required_certs):
                cert_names = ", ".join(str(c).split("#")[-1] for c in required_certs)
                return self._deny(student, f"Missing required certification: {cert_names}")

        # ---- CHECK 5: Duration limit ----
        max_dur = list(self.graph.objects(eq_uri, NS.maxDurationHours))
        if max_dur:
            limit = float(max_dur[0])
            if student.duration_hours > limit:
                return self._deny(student, f"Duration {student.duration_hours}h exceeds max {limit}h")

        # ---- CHECK 6: Time-slot conflict ----
        for res in self.graph.subjects(NS.reserves, eq_uri):
            for slot in self.graph.objects(res, NS.timeSlot):
                if str(slot) == student.time_slot:
                    return self._deny(student, f"Time slot {student.time_slot} already booked")

        # ---- ALL PASSED -> CREATE RESERVATION ----
        self.reservation_counter += 1
        res_uri = NS[f"Reservation{self.reservation_counter}"]
        self.graph.add((res_uri, RDF.type, NS.Reservation))
        self.graph.add((res_uri, NS.reserves, eq_uri))
        self.graph.add((res_uri, NS.reservedBy, stu_uri))
        self.graph.add((res_uri, NS.timeSlot, Literal(student.time_slot)))

        return self._approve(student, res_uri)

    def _approve(self, student, res_uri):
        msg = f"APPROVED {student.equipment_name} @ {student.time_slot}"
        self.log.append((student.unique_id, student.equipment_name, student.time_slot, "APPROVED"))
        return msg

    def _deny(self, student, reason):
        msg = f"DENIED {student.equipment_name} @ {student.time_slot} -> {reason}"
        self.log.append((student.unique_id, student.equipment_name, student.time_slot, f"DENIED: {reason}"))
        return msg

In [14]:
# Cell 6 (fixed) - LabBookingModel
class LabBookingModel(Model):
    def __init__(self, requests):
        super().__init__()

        # Load the ontology
        self.graph = Graph()
        self.graph.parse("lab_ontology.ttl", format="turtle")

        # Create the lab manager
        self.lab_manager = LabManagerAgent(self, unique_id=1, graph=self.graph)

        # Create student agents
        self.students = []
        for i, (stu, eq, slot, dur) in enumerate(requests):
            agent = StudentAgent(
                model=self,
                unique_id=i + 2,
                student_uri=NS[stu],
                equipment_name=eq,
                time_slot=slot,
                duration_hours=dur,
            )
            self.students.append(agent)

    def step(self):
        # One round: every agent acts once in random order
        self.agents.shuffle_do("step")

    def run_all(self):
        # Only ONE round is needed
        self.step()

In [15]:
# Cell 7 - Quick test of the whole simulation
test_requests = [
    ("Student1", "LaserCutter1",  "Mon10", 2),
    ("Student2", "LaserCutter1",  "Mon11", 2),
    ("Student2", "ChemAnalyzer1", "Mon10", 3),
    ("Student3", "Oscilloscope1", "Tue14", 1),
    ("Student1", "3DPrinter1",    "Tue14", 4),
    ("Student5", "Centrifuge1",   "Wed09", 10),
    ("Student1", "LaserCutter1",  "Mon10", 2),
]

model = LabBookingModel(test_requests)
model.run_all()

print("----- RESULTS -----")
for student in model.students:
    print(f"Student {student.unique_id}: {student.result}")

print("\n----- MANAGER LOG -----")
for entry in model.lab_manager.log:
    print(entry)

----- RESULTS -----
Student 2: APPROVED LaserCutter1 @ Mon10
Student 3: DENIED LaserCutter1 @ Mon11 -> Missing required certification: LaserSafety
Student 4: APPROVED ChemAnalyzer1 @ Mon10
Student 5: APPROVED Oscilloscope1 @ Tue14
Student 6: DENIED 3DPrinter1 @ Tue14 -> Equipment is under maintenance
Student 7: DENIED Centrifuge1 @ Wed09 -> Duration 10h exceeds max 2.0h
Student 8: DENIED LaserCutter1 @ Mon10 -> Time slot Mon10 already booked

----- MANAGER LOG -----
(7, 'Centrifuge1', 'Wed09', 'DENIED: Duration 10h exceeds max 2.0h')
(3, 'LaserCutter1', 'Mon11', 'DENIED: Missing required certification: LaserSafety')
(4, 'ChemAnalyzer1', 'Mon10', 'APPROVED')
(5, 'Oscilloscope1', 'Tue14', 'APPROVED')
(2, 'LaserCutter1', 'Mon10', 'APPROVED')
(8, 'LaserCutter1', 'Mon10', 'DENIED: Time slot Mon10 already booked')
(6, '3DPrinter1', 'Tue14', 'DENIED: Equipment is under maintenance')


In [ ]:
# Cell 8 - Launch the Streamlit app from inside Jupyter
!streamlit run app.py